In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00""
    ],
    "lat": [
        13.7, 14.2, 14.9
    ],
    "lon": [
        -33.2, -34.4, -35.5
    ],
    "pressure": [
        1009, 1009, 1009
    ],
    "wind": [
        25, 25, 25
    ],
    "stage": [
        "low", "low", "low"
    ]
}


In [ ]:
import pandas as pd

df = pd.DataFrame(track_data)
df["datetime"] = pd.to_datetime(df["datetime"])
print(df)

In [ ]:
ships_file = "/nfs/home11/ugrad/2024/es348865/honors_research/ships/edouard_2014.txt"

# Adjust depending on your file structure — if it’s comma-separated, use pd.read_csv(..., sep=",")
df_ships = pd.read_csv(ships_file, delim_whitespace=True, header=None, names=[
    "datetime", "SHRD", "SHDC"
])

# Convert datetime (format example: 2014091018)
df_ships["datetime"] = pd.to_datetime(df_ships["datetime"], format="%Y%m%d%H")
df_ships = df_ships.set_index("datetime")


In [ ]:
# Convert knots to m/s if you want physical units
df_ships["u_shear"] = -df_ships["SHRD"] * np.sin(np.deg2rad(df_ships["SHDC"]))
df_ships["v_shear"] = -df_ships["SHRD"] * np.cos(np.deg2rad(df_ships["SHDC"]))


In [ ]:
# Get nearest SHIPS data (same timestamp as storm_time)
if storm_time in df_ships.index:
    shear = df_ships.loc[storm_time]
else:
    # nearest value if no exact match
    shear = df_ships.iloc[(df_ships.index.get_indexer([storm_time], method='nearest'))[0]]

u = shear["u_shear"]
v = shear["v_shear"]
mag = shear["SHRD"]

# Add shear vector at bottom-left corner (lon_min + margin, lat_min + margin)
corner_lon = lon_min + 0.5
corner_lat = lat_min + 0.5

ax.quiver(
    corner_lon, corner_lat, u, v,
    angles='xy', scale_units='xy', scale=1.5,
    color='lime', width=0.005
)

# Label
ax.text(corner_lon + 0.3, corner_lat, f"{mag:.0f} kt shear", color='lime', fontsize=9)


In [ ]:
ax.quiverkey(ax.quiver(corner_lon, corner_lat, u, v),
             X=0.15, Y=0.05, U=10,
             label='10 kt shear', labelpos='E', color='lime')


In [ ]:
import pandas as pd
import xarray as xr
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import os
import datetime as dt

# =========================
# 0. Track data (2014 Edouard best track)
# =========================
track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        13.7, 14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -33.2, -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ]
}

df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

# =========================
# 1. Interpolate every 30 minutes
# =========================
interp_times = pd.date_range(start=df_track.index.min(),
                             end=df_track.index.max(),
                             freq='30min')

track_interp = df_track[["lat", "lon"]].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method='time')
track_interp = track_interp.loc[interp_times]

# =========================
# 2. Prepare output folder
# =========================
output_dir = "outputs_edouard_frames"
os.makedirs(output_dir, exist_ok=True)

# =========================
# 3. Load MERGIR and Lightning once
# =========================
merg_files = sorted(glob.glob("/spare11/atm350/es348865/MERGIR_data/merg_201409*.nc4"))
ds_merg = xr.open_mfdataset(merg_files, combine="by_coords")

lightning_files = sorted(glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv"))
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all['longitude'] = ((df_light_all['longitude'] + 180) % 360) - 180
df_light_all['datetime'] = pd.to_datetime(df_light_all['datetime'], format='%Y/%m/%d %H:%M:%S.%f')

# =========================
# =========================
# 4b. Extract SHIPS shear data into DataFrame
# =========================
import json
import datetime as dt
import matplotlib.pyplot as plt
from tropycal import tracks
basin = tracks.TrackDataset(basin='north_atlantic',include_btk=False)
storm = basin.get_storm(('edouard',2014))
ships = storm.get_ships(dt.datetime(2014, 9, 11, 12))
available_times = storm.search_ships()
valid_times = [ships.attrs['forecast_init'] + dt.timedelta(hours=int(h)) for h in ships.fhr]

df_ships = pd.DataFrame({
    "datetime": valid_times,
    "shear_mag": ships.shear_kt,
    "shear_dir": ships.shear_dir
})
df_ships["u_shear"] = -df_ships["shear_mag"] * np.sin(np.deg2rad(df_ships["shear_dir"]))
df_ships["v_shear"] = -df_ships["shear_mag"] * np.cos(np.deg2rad(df_ships["shear_dir"]))
df_ships = df_ships.set_index("datetime")

# =========================
# 5. Loop through interpolated times
# =========================
for storm_time in track_interp.index:
    print(f"Processing {storm_time}...")

    storm_lat = track_interp.loc[storm_time, "lat"]
    storm_lon = track_interp.loc[storm_time, "lon"]

    # Snap to MERGIR grid
    storm_lon_grid = ds_merg.lon.values[np.abs(ds_merg.lon.values - storm_lon).argmin()]
    storm_lat_grid = ds_merg.lat.values[np.abs(ds_merg.lat.values - storm_lat).argmin()]

    tb = ds_merg["Tb"].sel(time=storm_time, method="nearest").squeeze()

    # Subset region around storm
    lat_min, lat_max = storm_lat_grid - 6, storm_lat_grid + 6
    lon_min, lon_max = storm_lon_grid - 6, storm_lon_grid + 6
    tb_sub = tb.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

    # Lightning ±15 min window
    start = storm_time - pd.Timedelta(minutes=15)
    end = storm_time + pd.Timedelta(minutes=15)
    mask_time = (df_light_all['datetime'] >= start) & (df_light_all['datetime'] <= end)
    df_sub = df_light_all[
        mask_time &
        df_light_all["latitude"].between(lat_min, lat_max) &
        df_light_all["longitude"].between(lon_min, lon_max)
    ]

    print(f"Lightning strikes in ±15 min: {len(df_sub)}")

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 8))
    tb_sub.plot(cmap="turbo", vmin=180, vmax=300, add_colorbar=True, ax=ax)

    # Storm center + lightning
    ax.scatter(storm_lon_grid, storm_lat_grid, c="red", marker="x", s=100, label="Storm Center")
    ax.scatter(df_sub["longitude"], df_sub["latitude"], color="purple", s=4, alpha=0.6, label="Lightning")

    # Radius indicator
    circ = Circle((storm_lon_grid, storm_lat_grid), 200/60, edgecolor="black",
                  facecolor="none", linestyle="--", linewidth=1)
    ax.add_patch(circ)

    # --- Add SHIPS shear vector in bottom-left corner ---
    nearest_idx = df_ships.index.get_indexer([storm_time], method='nearest')[0]
    shear = df_ships.iloc[nearest_idx]
    u, v, mag = shear["u_shear"], shear["v_shear"], shear["shear_mag"]

    corner_lon = lon_min + 0.5
    corner_lat = lat_min + 0.5

    ax.quiver(
        corner_lon, corner_lat, u, v,
        angles='xy', scale_units='xy', scale=2.4,
        color='seagreen', width=0.006, headwidth=3, headlength=4
    )
    ax.text(
        corner_lon + 0.3, corner_lat + 0.2,
        f"{mag:.0f} kt shear", color='lime',
        fontsize=9, ha='left', va='bottom',
        weight='bold', alpha=0.8
    )

    # --- Style ---
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Hurricane Edouard - Brightness Temp + Lightning + SHIPS Shear\n{storm_time}")
    ax.legend(loc='upper right', fontsize=8, frameon=True)

    # --- Save frame ---
    outname = os.path.join(output_dir, f"edouard_overlay_{storm_time:%Y%m%d%H%M}.png")
    plt.savefig(outname, dpi=150, bbox_inches='tight')
    plt.close(fig)
